In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir bin

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mkdir: cannot create directory ‘bin’: File exists


Training a neural network is an iterative optimization process aimed at finding the parameter values that minimize the network's prediction error on the training data. This process relies on two main components: an initial setup and the repetitive forward/backward propagation cycle.
*   **Setup and core components ** steps involve defining the mathematical oundation and initial state of the network:1. Parameter Initialization ($\mathbf{W}, \mathbf{b}$):Goal: Randomly initialize the network's parameters (weights $\mathbf{W}$ and biases $\mathbf{b}$).Why: Starting with random, small values is crucial to break symmetry, ensuring different neurons learn different features.
*   **Forward Propagation Implementation** Goal: Implement the function that computes the network's output ($\hat{y}$) for a given input ($\mathbf{x}$).Mechanism: At each layer ($l$), the output $\mathbf{a}^{[l]}$ is calculated:$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$$$\mathbf{a}^{[l]} = g(\mathbf{z}^{[l]})$$where $g$ is the activation function.
*  **Cost Function Implementation** ($\mathcal{J}$):Goal: Implement the function that quantifies the difference (error) between the network's prediction ($\hat{y}$) and the true label ($y$).Purpose: The cost $\mathcal{J}$ is the value the training process seeks to minimize. Common examples include Cross-Entropy Loss for classification.
*  **Backpropagation Implementation:** Goal: Implement the algorithm to efficiently calculate the gradients (derivatives) of the cost function $\mathcal{J}$ with respect to every parameter ($\mathbf{W}$ and $\mathbf{b}$).Mechanism: It uses the chain rule to propagate the error backward through the network, determining how much each parameter contributed to the final error: $\frac{\partial \mathcal{J}}{\partial \mathbf{W}^{[l]}}$ and $\frac{\partial \mathcal{J}}{\partial \mathbf{b}^{[l]}}$.


Sequential Execution  

In [ ]:
!gcc -c /content/drive/MyDrive/DV2597_MP/assignment03/main.c -o ./main.o
!gcc -c /content/drive/MyDrive/DV2597_MP/assignment03/lenet.c -o ./lenet.o
!gcc ./main.o ./lenet.o -o /content/drive/MyDrive/DV2597_MP/assignment03/main_executable -lm

In [ ]:
import os

# Define the directory where the executable and data files are located
exe_dir = '/content/drive/MyDrive/DV2597_MP/assignment03/'
executable_name = 'main_executable'
executable_path = os.path.join(exe_dir, executable_name)

# Change the current working directory to exe_dir
# This ensures the executable can find its data files using relative paths
os.chdir(exe_dir)
print(f"Changed current working directory to: {os.getcwd()}")

# Grant execute permissions (already done, but safe to re-run)
!chmod +x {executable_name}

# Run the executable from the new current working directory
!time ./{executable_name}

Changed current working directory to: /content/drive/MyDrive/DV2597_MP/assignment03
Total images in training set: 60000
Batchsize: 300

Training on images: 0-300	
Training on images: 300-600	
Training on images: 600-900	Training  1% complete
Training on images: 900-1200	
Training on images: 1200-1500	Training  2% complete
Training on images: 1500-1800	
Training on images: 1800-2100	Training  3% complete
Training on images: 2100-2400	
Training on images: 2400-2700	Training  4% complete
Training on images: 2700-3000	
Training on images: 3000-3300	Training  5% complete
Training on images: 3300-3600	
Training on images: 3600-3900	Training  6% complete
Training on images: 3900-4200	
Training on images: 4200-4500	Training  7% complete
Training on images: 4500-4800	
Training on images: 4800-5100	Training  8% complete
Training on images: 5100-5400	
Training on images: 5400-5700	Training  9% complete
Training on images: 5700-6000	
Training on images: 6000-6300	Training 10% complete
Training on 

Parallel GPU execution  

In [ ]:
# 1. Compile the CUDA file
# We add -I to ensure it finds the header folder
!nvcc -arch=sm_75 -O3 -I/content/drive/MyDrive/DV2597_MP/assignment03/ \
    -c /content/drive/MyDrive/DV2597_MP/assignment03/lenet_par.cu -o ./lenet_par.o

# 2. Compile the Main file
!nvcc -arch=sm_75 -O3 -I/content/drive/MyDrive/DV2597_MP/assignment03/ \
    -c /content/drive/MyDrive/DV2597_MP/assignment03/main.c -o ./main.o

# 3. Link them together
!nvcc -arch=sm_75 ./main.o ./lenet_par.o -o ./main_cuda_executable -lm

/content/drive/MyDrive/DV2597_MP/assignment03/lenet_par.cu(573): warning #177-D: variable "outlen" was declared but never referenced
   const int outlen = (sizeof(features->output)/sizeof(double));
             ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

/content/drive/MyDrive/DV2597_MP/assignment03/main.c: In function ‘read_data’:
/content/drive/MyDrive/DV2597_MP/assignment03/main.c:46:9: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   46 |         fread(data, sizeof(*data)*count, 1, fp_image);
      |         ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/content/drive/MyDrive/DV2597_MP/assignment03/main.c:47:9: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]

In [ ]:
import os

# Define the directory where the executable and data files are located
exe_dir = '/content/drive/MyDrive/DV2597_MP/assignment03/'
executable_name = 'main_cuda_executable'
executable_path = os.path.join(exe_dir, executable_name)

# Change the current working directory to exe_dir
# This ensures the executable can find its data files using relative paths
os.chdir(exe_dir)
print(f"Changed current working directory to: {os.getcwd()}")

# Grant execute permissions (already done, but safe to re-run)
!chmod +x {executable_name}

# This tells CUDA to report the exact location of the crash
%env CUDA_LAUNCH_BLOCKING=1

# Now run your executable again
!./{executable_name}

Changed current working directory to: /content/drive/MyDrive/DV2597_MP/assignment03
env: CUDA_LAUNCH_BLOCKING=1
Total images in training set: 60000
Batchsize: 300

Training on images: 0-300	
Training on images: 300-600	
Training on images: 600-900	Training  1% complete
Training on images: 900-1200	
Training on images: 1200-1500	Training  2% complete
Training on images: 1500-1800	
Training on images: 1800-2100	Training  3% complete
Training on images: 2100-2400	
Training on images: 2400-2700	Training  4% complete
Training on images: 2700-3000	
Training on images: 3000-3300	Training  5% complete
Training on images: 3300-3600	
Training on images: 3600-3900	Training  6% complete
Training on images: 3900-4200	
Training on images: 4200-4500	Training  7% complete
Training on images: 4500-4800	
Training on images: 4800-5100	Training  8% complete
Training on images: 5100-5400	
Training on images: 5400-5700	Training  9% complete
Training on images: 5700-6000	
Training on images: 6000-6300	Traini